# 🎬 VIDEO-FREE · 5. Director: el video musical completo
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ivanherrerabaquero-dev/VIDEO-FREE/blob/main/colab/5_director_video.ipynb)

Este cuaderno hace **todo el video** a partir de tu canción y tus personajes:

1. **Plano de tomas.** Escribe la historia sobre los compases de la canción: dron de apertura, él se sube al Camaro, primeros planos cantando, la copiloto tarareando, manos en el volante, guitarra, recuerdo en blanco y negro, noche, fantasma y cierre.
2. **Fotogramas clave fotorrealistas** (RealVisXL). La cara de tu personaje se conserva **píxel por píxel** desde su recorte; la IA pinta el auto, la carretera y la luz alrededor.
3. **Movimiento real** (FastWan 2.2 en 3 pasos, o Wan 2.2 en 30). Cada fotograma cobra vida: camina, se sube al auto, maneja, el pelo con el viento, la cámara vuela.
4. **Labios** (LatentSync). En las tomas donde canta, la boca sigue la voz de la canción.
5. **Montaje.** Cortes al compás, color de cine según la hora, grano, franjas de cine y la canción.

| GPU | Borrador (480p) | Final |
|---|---|---|
| T4 gratis | ~30–50 min | ~45–75 min (576p) |
| T4 gratis + Kaggle (3 GPU) | ~15–25 min | ~20–35 min |
| A100 (pago) | ~8 min | ~15 min (704p) |

Tiempos aproximados para unos 30 s de canción, sin contar la primera descarga de los modelos.

- **Varias GPU gratis a la vez.** Este cuaderno (Colab, 1 GPU) y el cuaderno 7 (Kaggle, 2 GPU) se reparten las tomas. Pon `TRABAJADORES = 3` aquí y en Kaggle. Cada GPU sube sus tomas a tu repo (`medios/director/`) y esta monta el video cuando están todas.
- **Puedes cortar y seguir.** Cada fotograma y cada toma quedan en tu repo y en tu Drive. Si Colab se desconecta, vuelve a ejecutar: solo hace lo que falta.
- **LatentSync se guarda en tu Drive** la primera vez (si cabe). Las siguientes sesiones lo recuperan en 1–2 minutos en vez de instalarlo de nuevo.
- **Licencias:** RealVisXL y RealVisXL Lightning (OpenRAIL++), Wan 2.2 y FastWan 2.2 (Apache 2.0), LatentSync (Apache 2.0 / OpenRAIL++). Detalle en `colab/LICENCIAS-TERCEROS.md`.

**Antes de empezar (una sola vez):**
1. *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.*
2. Crea un token de GitHub (*Settings → Developer settings → Fine-grained tokens*) con acceso solo a **VIDEO-FREE** y permiso **Contents: Read and write**.
3. En Colab, abre el icono de **llave (Secretos)**, crea `GITHUB_TOKEN`, pega el token y activa *Acceso al cuaderno*.



In [ ]:
#@title 1 · Preparar Colab y conectar con tu repo
REPO = "ivanherrerabaquero-dev/VIDEO-FREE"  #@param {type:"string"}
RAMA = "main"  #@param {type:"string"}
#@markdown Guarda copias en tu Google Drive (Colab te pedirá permiso para acceder a Drive).
USAR_DRIVE = True  #@param {type:"boolean"}
import os, sys, subprocess
if os.path.exists("/content/VIDEO-FREE/.git"):
    subprocess.run(["git", "-C", "/content/VIDEO-FREE", "pull", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", RAMA, f"https://github.com/{REPO}", "/content/VIDEO-FREE"], check=True)
sys.path.insert(0, "/content/VIDEO-FREE/colab")
import importlib, comun; importlib.reload(comun)
info = comun.preparar(REPO, RAMA, usar_drive=USAR_DRIVE)
VRAM = info["gpu"].get("vram_gb", 0)

In [ ]:
#@title 2 · Instalar (2–3 min)
!pip install -q -U "diffusers>=0.35" transformers accelerate peft gguf ftfy sentencepiece imageio imageio-ffmpeg
import importlib, director, generar
importlib.reload(director); importlib.reload(generar)
print("Listo")

In [ ]:
#@title 3 · El plano de tomas y las opciones
PROYECTO = "carretera-sin-ti.json"  #@param {type:"string"}
CALIDAD = "borrador (rápido, 480p)"  #@param ["borrador (rápido, 480p)", "final (720p)"]
#@markdown Empieza con «borrador» para ver la historia completa rápido; luego haz «final».
FORMATO = "horizontal"  #@param ["horizontal", "vertical"]
MOTOR = "rápido (FastWan 2.2, 3 pasos)"  #@param ["rápido (FastWan 2.2, 3 pasos)", "calidad (Wan 2.2, 30 pasos)"]
MODO_PERSONAJE = "recorte (cara exacta)"  #@param ["recorte (cara exacta)", "entrenado (pose libre)"]
#@markdown «entrenado» usa los personajes del cuaderno 6.
LABIOS = True  #@param {type:"boolean"}
RITMO = "auto"  #@param ["auto", "rapido", "pausado"]
SEMILLA = 11  #@param {type:"integer"}
#@markdown **Varias GPU:** con Kaggle (cuaderno 7) pon `TRABAJADORES = 3`; esta GPU es el trabajador 1.
TRABAJADOR = 1  #@param {type:"integer"}
TRABAJADORES = 1  #@param {type:"integer"}
cfg = generar.Config(proyecto=PROYECTO, formato=FORMATO, calidad="borrador" if CALIDAD.startswith("borrador") else "final",
                     ritmo=RITMO, semilla=SEMILLA, motor="rapido" if MOTOR.startswith("rápido") else "calidad",
                     modo_personaje="entrenado" if MODO_PERSONAJE.startswith("entrenado") else "recorte",
                     labios=LABIOS, trabajador=TRABAJADOR, trabajadores=TRABAJADORES)
d = generar.Director(cfg, VRAM)
print(d.tabla())
print(f"\n{len(d.plano)} tomas · {sum(t['canta'] for t in d.plano)} con labios · a esta GPU le tocan: {[t['n'] for t in d.mias()]}")

In [ ]:
#@title 3b · (Opcional) Cambiar una toma
TOMA = 0  #@param {type:"integer"}
#@markdown Número de la toma en la tabla (0 = no cambiar nada). Deja vacío lo que no quieras cambiar. La toma se vuelve a hacer desde cero.
NUEVA_IMAGEN = ""  #@param {type:"string"}
#@markdown Qué se ve, en inglés (ej. `close-up of a young man singing in the rain inside a black 1969 convertible`).
NUEVO_MOVIMIENTO = ""  #@param {type:"string"}
#@markdown Qué pasa durante la toma, en inglés (ej. `he slams the steering wheel and shouts the lyrics`).
NUEVO_PERSONAJE = ""  #@param {type:"string"}
#@markdown Archivo de `medios/personajes/` (ej. `actor_canta.png`) o `ninguno`.
LABIOS_TOMA = "igual"  #@param ["igual", "sí", "no"]
NUEVA_SEMILLA = 0  #@param {type:"integer"}
import json
if TOMA:
    t = d.por_n[TOMA]
    if NUEVA_IMAGEN: t["imagen"] = f"{NUEVA_IMAGEN}, {director.ESTILO_FOTO}"
    if NUEVO_MOVIMIENTO: t["mueve"] = f"{NUEVO_MOVIMIENTO}, {director.MOVIMIENTO_COMUN}"
    if NUEVO_PERSONAJE:
        t["personaje"] = None if NUEVO_PERSONAJE == "ninguno" else NUEVO_PERSONAJE
        t["encuadre"] = t.get("encuadre") or "primer_plano"
    if LABIOS_TOMA != "igual": t["canta"] = LABIOS_TOMA == "sí"
    if NUEVA_SEMILLA: t["semilla"] = NUEVA_SEMILLA
    if TOMA not in cfg.rehacer: cfg.rehacer.append(TOMA)
    for f in (f"clave_{TOMA:02d}.jpg", f"toma_{TOMA:02d}.mp4", f"labios_{TOMA:02d}.mp4"):
        (d.local / f).unlink(missing_ok=True)
    comun.guardar(f"{d.base}/plano.json", json.dumps(d.plano, ensure_ascii=False, indent=1), f"Director: cambio la toma {TOMA}")
    print(director.tabla([t], d.p.get("letra")))

In [ ]:
#@title 4 · Fotogramas clave fotorrealistas (RealVisXL)
d.fotogramas()
print("¿Alguno no te gusta? Cámbialo en la celda 3b y vuelve a ejecutar desde aquí.")

In [ ]:
#@title 5 · Dar vida a cada toma (FastWan / Wan 2.2)
d.animar()

In [ ]:
#@title 6 · Labios que cantan la canción (LatentSync)
d.labios()

In [ ]:
#@title 7 · Montaje final con la canción
ESPERAR_OTRAS_GPU = 120  #@param {type:"integer"}
#@markdown Minutos que espera a las tomas de Kaggle (solo si TRABAJADORES > 1). Las que no lleguen quedan en negro.
FRANJAS_DE_CINE = True  #@param {type:"boolean"}
GRANO = 7  #@param {type:"slider", min:0, max:15, step:1}
video = d.montar(esperar_min=ESPERAR_OTRAS_GPU if cfg.trabajadores > 1 else 0, franjas=FRANJAS_DE_CINE, grano=GRANO)
comun.ver(video, 720)

In [ ]:
#@title 8 · Guardar en tu repo y poner las tomas en el proyecto
PONER_TOMAS_EN_ESCENAS = True  #@param {type:"boolean"}
#@markdown Cada toma pasa a ser una escena `video` del portal. Las escenas anteriores quedan en `escenas_antes_director`.
d.guardar(PONER_TOMAS_EN_ESCENAS)